# Lektion 13 - Agentenspeicher


## Einrichtung

Dieses Notebook zeigt, wie man einen Reisebuchungsagenten mit **persistentem Speicher** unter Verwendung des **Microsoft Agent Framework** (MAF) erstellt.

Sie lernen, wie verschiedene Arten von Agentenspeicher — Arbeits-, Kurzzeit- und Langzeitspeicher — beeinflussen, wie ein Agent Informationen über Gespräche hinweg behält und nutzt.

**Voraussetzungen:**
- Ein Microsoft Foundry-Projekt mit einem bereitgestellten Chat-Modell (z. B. `gpt-4.1-mini`).
- Angemeldet mit der Azure CLI — führen Sie `az login` in Ihrem Terminal aus.
- `AZURE_AI_PROJECT_ENDPOINT` — Ihr Microsoft Foundry-Projektendpunkt.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — der Name Ihres bereitgestellten Modells.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Arten von Agentenspeicher

KI-Agenten können verschiedene Arten von Speicher nutzen, von denen jede einem bestimmten Zweck dient:

### Arbeitsgedächtnis
Der Gesprächsverlauf selbst – die Nachrichten, die in einer Sitzung ausgetauscht werden. Der Agent kann sich auf frühere Nachrichten im gleichen Verlauf beziehen, um Kohärenz zu wahren. In MAF wird dies über **`agent.create_session()`** erstellt, was eine `AgentSession` zurückgibt.

### Kurzzeitgedächtnis
Informationen, die während der Dauer einer Aufgabe oder Sitzung bestehen bleiben, aber nicht dauerhaft gespeichert werden. Zum Beispiel kann der Agent während eines mehrstufigen Planungsgesprächs Fakten sammeln und diese verwenden, um eine endgültige Reiseroute zu erstellen.

### Langzeitgedächtnis
Präferenzen und Fakten, die **über Sitzungen hinweg** bestehen bleiben. Ein wiederkehrender Nutzer sollte seine Diätvorschriften oder seinen Reisestil nicht erneut angeben müssen. Langzeitgedächtnis wird typischerweise durch einen externen Speicher – eine Datenbank, Datei oder Vektorindex – unterstützt und dem Agenten über Tools zugänglich gemacht.


## Arbeitsgedächtnis mit Sitzungen

Die einfachste Form des Gedächtnisses ist die Gesprächssitzung. Wenn Sie dieselbe Sitzung (erstellt über `agent.create_session()`) auf nachfolgende `agent.run()`-Aufrufe anwenden, sieht der Agent die vollständige Historie dieses Gesprächs und kann sich an frühere Details erinnern.

Lassen Sie uns einen Reiseberater erstellen und das Arbeitsgedächtnis demonstrieren.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Der Agent erinnerte sich korrekt an das Budget, weil beide Nachrichten dieselbe Sitzung teilen. Das ist **Arbeitsgedächtnis** — es existiert nur für die Dauer der Sitzung.

### Was passiert bei einem neuen Thread?

Wenn wir eine **neue** Sitzung erstellen, hat der Agent keine Erinnerung an das vorherige Gespräch:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Langzeitgedächtnis-Muster

Um Benutzerpräferenzen **über Sitzungen hinweg** zu speichern, benötigen wir einen persistenten Speicher, der außerhalb des Konversationsverlaufs lebt. Der Agent greift über **Tools** darauf zu — Funktionen, die er zum Speichern und Abrufen von Informationen aufrufen kann.

Unten implementieren wir einen einfachen In-Memory-Präferenzspeicher (im Produktiveinsatz würde man diesen mit einer Datenbank oder einem Vektorindex absichern) und stellen ihn als Tools bereit, die der Agent nutzen kann.

### Architektur
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Szenario 1 — Erstmaliger Nutzer bucht eine Jubiläumsreise

Sarah besucht die Website zum ersten Mal. Der Agent sollte ihre Vorlieben über die Tools speichern und diese nutzen, um Hotels zu empfehlen.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Szenario 2 — Sarah kehrt Wochen später zurück

Sarah startet einen **ganz neuen Thread** (simuliert eine neue Sitzung). Das Arbeitsgedächtnis ist leer, aber der langfristige Präferenzspeicher enthält noch ihre Informationen. Der Agent sollte diese abrufen und zur Personalisierung der Empfehlungen verwenden.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Zusammenfassung

In dieser Lektion haben Sie drei Arten von Agentengedächtnis kennengelernt und wie man sie mit dem Microsoft Agent Framework implementiert:

| Gedächtnistyp | MAF-Mechanismus | Lebensdauer |
|---|---|---|
| **Arbeitsgedächtnis** | `agent.create_session()` | Einzelne Unterhaltung |
| **Kurzzeitgedächtnis** | Akkumulierter Kontext innerhalb eines Gesprächs | Einzelne Aufgabe / Sitzung |
| **Langzeitgedächtnis** | Externer Speicher, zugänglich über `@tool`-Funktionen | Über Sitzungen hinweg |

### Wichtige Erkenntnisse
1. **`agent.create_session()`** bietet das Arbeitsgedächtnis — der Agent sieht die gesamte Gesprächshistorie innerhalb einer Sitzung.
2. **Neue Sitzungen verlieren den Kontext** — ohne Langzeitgedächtnis kann sich der Agent nicht an vergangene Gespräche erinnern.
3. **`@tool`-Funktionen** überbrücken die Lücke — sie ermöglichen es dem Agenten, Informationen in einem persistenten Speicher zu speichern und abzurufen.
4. **Personalisierung verbessert sich mit der Zeit** — je mehr Präferenzen gespeichert werden, desto besser werden die Empfehlungen des Agenten.

### Anwendungen in der Praxis
- **Kundendienst**: Kundenhistorie und -präferenzen merken
- **Persönliche Assistenten**: Kontext über Tage oder Wochen erhalten
- **Gesundheitswesen**: Patienteninformationen und Präferenzen verfolgen
- **E-Commerce**: Personalisierte Einkaufsangebote basierend auf der Historie

### Nächste Schritte
- Ersetzen Sie das In-Memory-Dict durch eine Datenbank oder einen Vektorspeicher (z.B. Azure AI Search)
- Fügen Sie eine Gedächtnisablaufzeit für zeitkritische Informationen hinzu
- Bauen Sie Multi-Agenten-Systeme mit gemeinsam genutztem Gedächtnis
- Erkunden Sie das Cognee-Notizbuch für ein wissensgraphgestütztes Gedächtnis


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
